In [ ]:
import pandas as pd
import tensorflow as tf
import sentencepiece as spm
import json
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [ ]:
english_tokenizer = spm.SentencePieceProcessor(model_file='/kaggle/input/protein_ai_tokenizers/tensorflow2/two_spm/1/tokenizers_gene_ai/english.model')
protein_tokenizer = spm.SentencePieceProcessor(model_file='/kaggle/input/protein_ai_tokenizers/tensorflow2/two_spm/1/tokenizers_gene_ai/protein.model')

In [ ]:
def make_ds(csv_path, protein_tokenizer, english_tokenizer):
    df = pd.read_csv(csv_path)
    # Tokenize protein sequences
    df["input_ids"] = df["Sequence"].apply(lambda x: protein_tokenizer.encode(x, out_type=int, add_bos=True, add_eos=True))
    
    # Tokenize outputs (text)
    df["output_ids"] = df["Outputs"].apply(lambda x: english_tokenizer.encode(x, out_type=int, add_bos=True, add_eos=True))

    max_input_len = 1024   # or choose based on your dataset
    max_output_len = 512
    
    df["input_ids"] = list(pad_sequences(df["input_ids"], maxlen=max_input_len, padding="post"))
    df["output_ids"] = list(pad_sequences(df["output_ids"], maxlen=max_output_len, padding="post"))

    y = np.array(df["input_ids"].tolist(), dtype=np.int32)
    X = np.array(df["output_ids"].tolist(), dtype=np.int32)

    y[y == -1] = 0

    decoder_input = y[:, :-1]  # all tokens except the last
    decoder_target = y[:, 1:]  # all tokens except the first

    dataset = tf.data.Dataset.from_tensor_slices(((X, decoder_input), decoder_target))

    BATCH_SIZE = 32

    dataset = dataset.shuffle(buffer_size=len(df))
    
    train_size = int(0.8 * len(df))
    train_ds = dataset.take(train_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_ds = dataset.skip(train_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


    return train_ds, val_ds

In [ ]:
train_ds, val_ds = make_ds("/kaggle/input/human-protein-dataset/dataset (1).csv", protein_tokenizer, english_tokenizer)

In [ ]:
def positional_encoding(length, depth):
  depth = depth/2

  positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
  depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)

  angle_rates = 1 / (10000**depths)         # (1, depth)
  angle_rads = positions * angle_rates      # (pos, depth)

  pos_encoding = np.concatenate(
      [np.sin(angle_rads), np.cos(angle_rads)],
      axis=-1) 

  return tf.cast(pos_encoding, dtype=tf.float32)

In [ ]:
class PositionalEmbedding(tf.keras.layers.Layer):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.d_model = d_model
    self.embedding = tf.keras.layers.Embedding(vocab_size, d_model, mask_zero=True) 
    self.pos_encoding = positional_encoding(length=2048, depth=d_model)

  def compute_mask(self, *args, **kwargs):
    return self.embedding.compute_mask(*args, **kwargs)

  def call(self, x):
    length = tf.shape(x)[1]
    x = self.embedding(x)
    # This factor sets the relative scale of the embedding and positonal_encoding.
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x = x + self.pos_encoding[tf.newaxis, :length, :]
    return x

In [ ]:
class BaseAttention(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

In [ ]:
class CrossAttention(BaseAttention):
  def call(self, x, context):
    attn_output, attn_scores = self.mha(
        query=x,
        key=context,
        value=context,
        return_attention_scores=True)

    # Cache the attention scores for plotting later.
    self.last_attn_scores = attn_scores

    x = self.add([x, attn_output])
    x = self.layernorm(x)

    return x

In [ ]:
class GlobalSelfAttention(BaseAttention):
  def call(self, x):
    attn_output = self.mha(
        query=x,
        value=x,
        key=x)
    x = self.add([x, attn_output])
    x = self.layernorm(x)
    return x

In [ ]:
class CausalSelfAttention(BaseAttention):
  def call(self, x):
    attn_output = self.mha(
        query=x,
        value=x,
        key=x,
        use_causal_mask = True)
    x = self.add([x, attn_output])
    x = self.layernorm(x)
    return x

In [ ]:
class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dff, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x) 
    return x

In [ ]:
class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()

    self.self_attention = GlobalSelfAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x):
    x = self.self_attention(x)
    x = self.ffn(x)
    return x

In [ ]:
class Encoder(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads,
               dff, vocab_size, dropout_rate=0.1):
    super().__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    self.pos_embedding = PositionalEmbedding(
        vocab_size=vocab_size, d_model=d_model)

    self.enc_layers = [
        EncoderLayer(d_model=d_model,
                     num_heads=num_heads,
                     dff=dff,
                     dropout_rate=dropout_rate)
        for _ in range(num_layers)]
    self.dropout = tf.keras.layers.Dropout(dropout_rate)

  def call(self, x):
    # `x` is token-IDs shape: (batch, seq_len)
    x = self.pos_embedding(x)  # Shape `(batch_size, seq_len, d_model)`.

    # Add dropout.
    x = self.dropout(x)

    for i in range(self.num_layers):
      x = self.enc_layers[i](x)

    return x  # Shape `(batch_size, seq_len, d_model)`.

In [ ]:
class DecoderLayer(tf.keras.layers.Layer):
  def __init__(self,
               *,
               d_model,
               num_heads,
               dff,
               dropout_rate=0.1):
    super(DecoderLayer, self).__init__()

    self.causal_self_attention = CausalSelfAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.cross_attention = CrossAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x, context):
    x = self.causal_self_attention(x=x)
    x = self.cross_attention(x=x, context=context)

    # Cache the last attention scores for plotting later
    self.last_attn_scores = self.cross_attention.last_attn_scores

    x = self.ffn(x)  # Shape `(batch_size, seq_len, d_model)`.
    return x

In [ ]:
class Decoder(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads, dff, vocab_size,
               dropout_rate=0.1):
    super(Decoder, self).__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    self.pos_embedding = PositionalEmbedding(vocab_size=vocab_size,
                                             d_model=d_model)
    self.dropout = tf.keras.layers.Dropout(dropout_rate)
    self.dec_layers = [
        DecoderLayer(d_model=d_model, num_heads=num_heads,
                     dff=dff, dropout_rate=dropout_rate)
        for _ in range(num_layers)]

    self.last_attn_scores = None

  def call(self, x, context):
    # `x` is token-IDs shape (batch, target_seq_len)
    x = self.pos_embedding(x)  # (batch_size, target_seq_len, d_model)

    x = self.dropout(x)

    for i in range(self.num_layers):
      x  = self.dec_layers[i](x, context)

    self.last_attn_scores = self.dec_layers[-1].last_attn_scores

    # The shape of x is (batch_size, target_seq_len, d_model).
    return x

In [ ]:
class Transformer(tf.keras.Model):
  def __init__(self, *, num_layers, d_model, num_heads, dff,
               input_vocab_size, target_vocab_size, dropout_rate=0.1):
    super().__init__()
    self.encoder = Encoder(num_layers=num_layers, d_model=d_model,
                           num_heads=num_heads, dff=dff,
                           vocab_size=input_vocab_size,
                           dropout_rate=dropout_rate)

    self.decoder = Decoder(num_layers=num_layers, d_model=d_model,
                           num_heads=num_heads, dff=dff,
                           vocab_size=target_vocab_size,
                           dropout_rate=dropout_rate)

    self.final_layer = tf.keras.layers.Dense(target_vocab_size)

  def call(self, inputs):
    # To use a Keras model with `.fit` you must pass all your inputs in the
    # first argument.
    context, x  = inputs

    context = self.encoder(context)  # (batch_size, context_len, d_model)

    x = self.decoder(x, context)  # (batch_size, target_len, d_model)

    # Final linear layer output.
    logits = self.final_layer(x)  # (batch_size, target_len, target_vocab_size)

    try:
      # Drop the keras mask, so it doesn't scale the losses/metrics.
      # b/250038731
      del logits._keras_mask
    except AttributeError:
      pass

    # Return the final output and the attention weights.
    return logits

In [ ]:
num_layers = 4
d_model = 128
dff = 512
num_heads = 8
dropout_rate = 0.1

In [ ]:
transformer = Transformer(
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    dff=dff,
    input_vocab_size=english_tokenizer.get_piece_size(),
    target_vocab_size=protein_tokenizer.get_piece_size(),
    dropout_rate=dropout_rate)

In [ ]:
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
  def __init__(self, d_model, warmup_steps=4000):
    super().__init__()

    self.d_model = d_model
    self.d_model = tf.cast(self.d_model, tf.float32)

    self.warmup_steps = warmup_steps

  def __call__(self, step):
    step = tf.cast(step, dtype=tf.float32)
    arg1 = tf.math.rsqrt(step)
    arg2 = step * (self.warmup_steps ** -1.5)

    return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

In [ ]:
learning_rate = CustomSchedule(d_model)

optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98,
                                     epsilon=1e-9)

In [ ]:
def masked_loss(label, pred):
  mask = label != 0
  loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')
  loss = loss_object(label, pred)

  mask = tf.cast(mask, dtype=loss.dtype)
  loss *= mask

  loss = tf.reduce_sum(loss)/tf.reduce_sum(mask)
  return loss


def masked_accuracy(label, pred):
  pred = tf.argmax(pred, axis=2)
  label = tf.cast(label, pred.dtype)
  match = label == pred

  mask = label != 0

  match = match & mask

  match = tf.cast(match, dtype=tf.float32)
  mask = tf.cast(mask, dtype=tf.float32)
  return tf.reduce_sum(match)/tf.reduce_sum(mask)

In [ ]:
transformer.compile(
    loss=masked_loss,
    optimizer=optimizer,
    metrics=[masked_accuracy])

In [ ]:
transformer.fit(train_ds,
                epochs=10,
                validation_data=val_ds)

In [ ]:
class ProteinTranslator(tf.Module):
    def __init__(self, tokenizers, transformer):
        """
        tokenizers: dict with 'protein' and 'text' keys
            - tokenizers['protein']: Protein_Tokenizer instance
            - tokenizers['text']: SentencePieceProcessor instance
        transformer: your seq2seq Transformer model
        """
        self.tokenizers = tokenizers
        self.transformer = transformer

    def __call__(self, sequence, max_length=512):
        """
        Autoregressive inference:
        - sequence: protein sequence string
        - max_length: max length of generated text
        """
        # --- Encode protein input ---
        if isinstance(sequence, str):
            seq_tokens = self.tokenizers['text'].encode(sequence, out_type=int, add_bos=True, add_eos=True)
            encoder_input = tf.convert_to_tensor([seq_tokens], dtype=tf.int32)  # shape (1, seq_len)
        else:
            raise ValueError("sequence must be a string")

        # --- Initialize decoder input with <BOS> ---
        BOS_ID = 1  # from your SentencePiece tokenizer
        EOS_ID = 2

        output_array = tf.TensorArray(dtype=tf.int32, size=0, dynamic_size=True)
        output_array = output_array.write(0, BOS_ID)

        # --- Autoregressive loop ---
        for i in tf.range(max_length):
            decoder_input = tf.transpose(output_array.stack())[tf.newaxis, :]  # shape (1, seq_len)
            
            predictions = self.transformer([encoder_input, decoder_input], training=False)
            # last token predictions
            predictions = predictions[:, -1, :]  # shape (1, vocab_size)
            predicted_id = tf.argmax(predictions, axis=-1, output_type=tf.int32)

            output_array = output_array.write(i+1, predicted_id[0])

            if predicted_id[0] == EOS_ID:
                break

        # --- Convert output tensor to sequence ---
        output_seq = tf.transpose(output_array.stack())[tf.newaxis, :]  # shape (1, tokens)

        # Decode to text using SentencePiece
        tokens_list = output_seq.numpy().tolist()[0]
        text = self.tokenizers['protein'].decode(tokens_list)

        return text, tokens_list

In [ ]:
tokenizers = {
    'protein': protein_tokenizer,  # your Protein_Tokenizer instance
    'text': english_tokenizer                     # your SentencePieceProcessor
}

translator = ProteinTranslator(tokenizers, transformer)

protein_seq = "Organism: Homo sapiens (Human) Protein_name: Protein PIGBOS1 (PIGB opposite strand protein 1) Molecular Function: nan Biological Process: regulation of endoplasmic reticulum unfolded protein response [GO:1900101]; response to unfolded protein [GO:0006986] Cellular Component: mitochondrial outer membrane [GO:0005741]; mitochondrion [GO:0005739]" #human insulin P01308
text, tokens = translator(protein_seq)
print("Generated text:", text)
print("Token IDs:", tokens)